Some Definitions:
Depth: Quantity of orders available in each sepcific level
BId depth: quantity of people who want to buy at each pice
Ask depth: quantity of people who want to sell at each price 

TIck: individual event in the market as update of the book of transaction 
Event: 


The bid price and size represent the demand for a stock, while the ask price and size rep
resent the supply. We enumerate these observations by n and compare Between two such observations, only one of the following events can occur

- $P_n^B > P_{n-1}^B$ or $q_n^B > q_{n-1}^B$, signifying an increase in demand.
- $P_n^B < P_{n-1}^B$ or $q_n^B < q_{n-1}^B$, signifying a decrease in demand.
- $P_n^A < P_{n-1}^A$ or $q_n^A > q_{n-1}^A$, signifying an increase in supply.
- $P_n^A > P_{n-1}^A$ or $q_n^A < q_{n-1}^A$, signifying a decrease in supply.



$$
e_n =
\mathbf{I}_{\{P_n^B \geq P_{n-1}^B\}} q_n^B
-
\mathbf{I}_{\{P_n^B \leq P_{n-1}^B\}} q_{n-1}^B
-
\mathbf{I}_{\{P_n^A \leq P_{n-1}^A\}} q_n^A
+
\mathbf{I}_{\{P_n^A \geq P_{n-1}^A\}} q_{n-1}^A
$$



If we have more pressure on the sales e < 0 if more pressure on buys e > 0 if q^B increases but P^B remains teh same e_n = q_n^B - q_n-1^B because both conditions are true (P_n^B == P_n-1^B) in the bid part of e_n in other words the model measuer the net change on quantity to this price this may say the pressure of the bid but does not mean that the price will increase 



*Indicator Function*
$$
I_{\{\text{condition}\}} =
\begin{cases}
1 & \text{if the condition is true} \\
0 & \text{if the condition is false}
\end{cases}
$$


In [2]:

import pandas as pd

df = pd.DataFrame({
    "bid_price": [100, 100, 101, 101, 100],
    "bid_size":  [5,   8,   4,   6,   3],
    "ask_price": [102, 102, 103, 103, 102],
    "ask_size":  [7,   6,   5,   8,   9],
})

df.index.name = "observation"
df


,bid_price,bid_size,ask_price,ask_size
observation,,,,
0,100,5,102,7
1,100,8,102,6
2,101,4,103,5
3,101,6,103,8
4,100,3,102,9


In [3]:

prev = df.shift(1)

df["ofi_event"] = (
    (df["bid_price"] >= prev["bid_price"]).astype(int) *df["bid_size"]

    - (df["bid_price"] <= prev["bid_price"]).astype(int)* prev["bid_size"]

    - (df["ask_price"] <= prev["ask_price"]).astype(int)* df["ask_size"]

    + (df["ask_price"] >= prev["ask_price"]).astype(int)* prev["ask_size"]
)


print(df)

             bid_price  bid_size  ask_price  ask_size  ofi_event
observation                                                     
0                  100         5        102         7        NaN
1                  100         8        102         6        4.0
2                  101         4        103         5       10.0
3                  101         6        103         8       -1.0
4                  100         3        102         9      -15.0


We have determined the e_n the conbtribution of a transition between two snapshos of the book of orders but the events occurts in irregular times many in one seconds and less in the next second, so we have

$\tau_n$   time instan where n event occurs

$N(t)$ Numbers of events occured until t $t$

$e_n$ OFI contribution to event n $n$


N(t) is defined as


$N(t) = max\{n | \tau_n <= t\}$

So the OFI in the interval k is the sum of contributtions of all events that occurred in the interval

$$\mathrm{OFI}_k = \sum_{n= N(t_{k-1})+1}^{N(t_k)} e_n $$

In [5]:
df["timestamp"] = pd.date_range(
    start = "2026-01-01 10:00:00",
    periods = len(df),
    freq = "1s"
)
df[["timestamp", "ofi_event"]] 
df = df.set_index("timestamp") ##newtempindex 
ofi_2s = df["ofi_event"].resample("2s").sum(min_count=1) ##groups by 2 sec and then sum from 1 
ofi_2s

timestamp
2026-01-01 10:00:00     4.0
2026-01-01 10:00:02     9.0
2026-01-01 10:00:04   -15.0
Freq: 2s, Name: ofi_event, dtype: float64

Mid Price is the midprice in instante t_k

$$\Delta P_k = \frac{(P_k) - (P-{k_1})}{\delta}$$

$p_k$ = mid quote price at time $t_k$

$\delta$ is the tick size


Long story short the empirical find shows
$$\Delta P_k = \hat{\alpha}_i + \hat{\beta}_i OFI_k + \hat{\epsilon}_k$$


$\Delta P_k$: Change of mid price in interval k $k$.



$OFI_k$: Cummulated OFI on interval k



$\hat{\alpha}$: intercept 



$\hat{\beta}$: price sensibility 



$\hat{\epsilon}_k$: error